# Does the app give the same answers as the tested notebook?

The app (`engine/core.py`) is meant to make exactly the same decisions as `P2_churn_engine_v3.ipynb`. This notebook checks that on any dataset, and, if you have the answer key for a synthetic dataset, also scores how well the app did.

**Steps**
1. Open the app, upload the dataset in the sidebar, and click **Ranked list of all merchants** (bottom of the Workbench tab). You get `ranked_merchants.csv`.
2. Run `P2_churn_engine_v3.ipynb` on the same dataset (set `DATA_FILE` in section 0). You get `P2_engine_v3_results_<dataset name>.xlsx`.
3. Put both files (and the truth file, if you have one) next to this notebook, set the three names below, and run all cells.

Nothing here is tuned: it only reads the two outputs and compares them.

In [ ]:
import os, numpy as np, pandas as pd
pd.set_option('display.width', 200); pd.set_option('display.max_colwidth', 80)

APP_EXPORT = 'ranked_merchants.csv'                                             # downloaded from the app
NOTEBOOK_RESULTS = 'P2_engine_v3_results_Synthetic_P2_dataset_seed20261115.xlsx'  # written by P2_churn_engine_v3.ipynb
TRUTH = 'Synthetic_P2_truth_seed20261115.xlsx'                                  # answer key for a synthetic dataset; set to None if you have none

for f in [APP_EXPORT, NOTEBOOK_RESULTS] + ([TRUTH] if TRUTH else []):
    if not os.path.exists(f):
        try:
            from google.colab import files
            print(f'Upload {f}'); files.upload()
        except ImportError:
            raise FileNotFoundError(f'Put {f} next to this notebook.')
app = pd.read_csv(APP_EXPORT, encoding='utf-8-sig').set_index('merchant_id')
nbk = pd.read_excel(NOTEBOOK_RESULTS, sheet_name='All merchants (ranked)').set_index('merchant_id')
print(f'App: {len(app)} merchants | Notebook: {len(nbk)} merchants | in both: {len(app.index.intersection(nbk.index))}')

## 1. Same decisions?

In [ ]:
def norm(s):
    s = s.copy()
    if s.dtype == bool: return s.astype(str).str.lower()
    return s.fillna('').astype(str).str.strip().str.split(':').str[0].replace({'nan': '', 'None': ''}).str.lower()

ids = nbk.index.intersection(app.index)
checks = [('Risk tier', 'tier'), ('Main driver', 'driver_main'), ('Second driver', 'driver_second'), ('Likely bucket (unclear cause)', 'review_likely_bucket'),
          ('Needs a person', 'needs_person'), ('Playbook', 'playbook'), ('Second playbook', 'playbook_also'), ('Rank', 'revenue_rank'),
          ('Call-list slot (Firm / Borderline / Next in line)', 'call_list_status'), ('Call this week', 'call_this_week'),
          ('Account-manager alert', 'am_alert'), ('Automated track', 'automated_track'), ('Expected saving (Rs)', 'expected_saving_rs')]
rows = []
for label, col in checks:
    if col not in app or col not in nbk:
        rows.append((label, 'column missing', '')); continue
    a, b = app.loc[ids, col], nbk.loc[ids, col]
    if col in ('revenue_rank', 'expected_saving_rs'):
        diff = ~np.isclose(pd.to_numeric(a, errors='coerce').fillna(-1), pd.to_numeric(b, errors='coerce').fillna(-1), atol=1)
    else:
        diff = norm(a) != norm(b)
    rows.append((label, f'{int((~diff).sum())} of {len(ids)}', ', '.join(ids[diff][:10])))
same = pd.DataFrame(rows, columns=['decision', 'same in app and notebook', 'merchants that differ'])
display(same)
calls_app = set(app.index[app.revenue_rank <= 15]); calls_nb = set(nbk.index[nbk.revenue_rank <= 15])
print(f'Same 15 calls: {calls_app == calls_nb}  ({len(calls_app & calls_nb)} of 15 shared)')
all_same = (same['merchants that differ'] == '').all() and calls_app == calls_nb
print('VERDICT:', 'the app and the notebook give identical decisions.' if all_same else 'there are differences: see the table above.')

## 2. How well did the app do? (only for a synthetic dataset with an answer key)

**True churner** = the merchant's volume in month 9 fell below half its healthy level (the brief's definition), which the app never saw.
**Right playbook** = the playbook for the problem that started first (pricing PB-3, performance PB-1, service PB-2, engagement PB-4), counting the second playbook too.
A merchant that left for no visible reason counts as right if it was sent to a person. A merchant that would not have left is a false alarm.

In [ ]:
if TRUTH:
    t = pd.read_excel(TRUTH).set_index('merchant_id'); d = app.join(t, how='inner')
    y = d.true_churn_90d.astype(bool)
    flagged = d.tier.astype(str).str[0].isin(['1', '2', '3']) & ~d.tier.astype(str).str.startswith('3P')
    calls = d.revenue_rank <= 15
    FAM = ['performance', 'service', 'pricing', 'engagement']; PBID = {'performance': 'PB-1', 'service': 'PB-2', 'pricing': 'PB-3', 'engagement': 'PB-4'}
    def first_problem(r):
        if r.churn_kind == 'switch at renewal': return 'pricing'
        found = {k: r[f'onset_{k}'] for k in FAM if r[f'has_{k}']}
        return min(found, key=found.get) if found else 'none'
    d['first_problem'] = d.apply(first_problem, axis=1)
    def playbook_right(r):
        if not r.churn_intent: return False
        if r.first_problem == 'none': return str(r.needs_person).lower() == 'true'
        return PBID[r.first_problem] in (r.playbook_id, r.playbook_also_id)
    d['playbook_right'] = d.apply(playbook_right, axis=1)
    named = d[flagged & y & d.driver_main.isin(['performance', 'service', 'engagement'])]
    score = pd.DataFrame([
        ('True churners in the 15 calls', f'{int((calls & y).sum())} of {int(calls.sum())}'),
        ('Precision of the at-risk group (tiers 1-3)', f'{(flagged & y).sum() / max(flagged.sum(), 1):.0%} ({int((flagged & y).sum())} of {int(flagged.sum())})'),
        ('Recall: true churners found (tiers 1-3)', f'{(flagged & y).sum() / y.sum():.0%} ({int((flagged & y).sum())} of {int(y.sum())})'),
        ('Driver = the problem that started first', f'{(named.driver_main == named.first_problem).mean():.0%} of {len(named)}'),
        ('Driver = a problem the merchant had', f"{named.apply(lambda r: bool(r['has_' + r.driver_main]), axis=1).mean():.0%} of {len(named)}"),
        ('Playbook right, flagged merchants', f"{d[flagged].playbook_right.mean():.0%} ({int(d[flagged].playbook_right.sum())} of {int(flagged.sum())})"),
        ('Seasonal decoys wrongly flagged', f"{int((flagged & d.case.str.startswith('Decoy: seasonal')).sum())} of {int(d.case.str.startswith('Decoy: seasonal').sum())}")],
        columns=['measure', 'result'])
    display(score)
    print('Playbook misses:'); display(d[flagged & ~d.playbook_right][['case', 'first_problem', 'driver_main', 'playbook_id', 'playbook_also_id', 'needs_person']])